# Lorenz Attractor

## Background

The **Lorenz system** (Edward Lorenz, 1963) is the most famous example of deterministic chaos. Originally derived as a simplified model of atmospheric convection, it became the cornerstone of chaos theory. In reservoir computing it is used to demonstrate that a reservoir can learn and reproduce the long-term attractor structure of a chaotic system.

Used in QRC papers:
- Fujii & Nakajima (2020) -- *QRC for near-term quantum devices* (Lorenz attractor reconstruction)
- Gauthier et al. (2021) -- *Next-generation reservoir computing* (autonomous Lorenz prediction)
- Tanaka et al. (2019) -- *Physical reservoir computing review*

## The Equations

dx/dt = sigma * (y - x)
dy/dt = x * (rho - z) - y
dz/dt = x * y - beta * z

Standard chaotic parameters (Lorenz 1963):

| Parameter | Value | Meaning |
|-----------|-------|--------|
| sigma     | 10.0  | Prandtl number |
| rho       | 28.0  | Rayleigh number |
| beta      | 8/3   | geometric factor |

With these parameters the system has **two strange attractors** (the butterfly wings) and a **Lyapunov exponent ~0.9** -- very sensitive to initial conditions.

## Why it is ideal for QRC

1. **Multivariate**: 3 coupled variables (x, y, z) -- reservoir sees one, must predict others
2. **Truly chaotic**: Lyapunov time ~1/0.9 ≈ 1.1 time units -- tests short-horizon prediction
3. **Attractor geometry**: even if exact prediction fails, reservoir should reproduce the butterfly shape
4. **Autonomous forecasting**: train reservoir to predict next state, then run it autonomously (closed loop) and check if attractor is reproduced -- a key test in Gauthier 2021

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from scipy.integrate import solve_ivp

np.random.seed(42)

## Numerical Integration

We use scipy's RK45 (Runge-Kutta 4th/5th order adaptive solver) for accurate integration of this stiff chaotic ODE.

In [ ]:
def lorenz_rhs(t, state, sigma=10.0, rho=28.0, beta=8/3):
    """Right-hand side of the Lorenz system."""
    x, y, z = state
    return [
        sigma * (y - x),
        x * (rho - z) - y,
        x * y - beta * z
    ]


def generate_lorenz(
    T_total=100.0,
    dt=0.02,
    x0=(0.0, 1.0, 1.05),
    sigma=10.0, rho=28.0, beta=8/3,
    washout_time=10.0,
):
    """
    Integrate the Lorenz system and return the trajectory.

    Parameters
    ----------
    T_total      : total integration time
    dt           : output timestep (observation interval)
    x0           : initial condition (x, y, z)
    sigma, rho, beta : Lorenz parameters
    washout_time : transient time to discard

    Returns
    -------
    t_out  : np.ndarray, shape (N,)
    xyz    : np.ndarray, shape (N, 3)
    """
    t_span = (0, T_total + washout_time)
    t_eval = np.arange(0, T_total + washout_time, dt)

    sol = solve_ivp(
        lorenz_rhs,
        t_span,
        x0,
        method='RK45',
        t_eval=t_eval,
        args=(sigma, rho, beta),
        rtol=1e-9,
        atol=1e-9,
    )

    washout_idx = int(washout_time / dt)
    xyz = sol.y[:, washout_idx:].T   # shape (N, 3)
    t_out = sol.t[washout_idx:] - washout_time
    return t_out, xyz


t, xyz = generate_lorenz(T_total=120.0, dt=0.02)
x_lor, y_lor, z_lor = xyz[:, 0], xyz[:, 1], xyz[:, 2]

print(f'Trajectory shape: {xyz.shape}')
print(f'Time span: {t[0]:.1f} to {t[-1]:.1f} (dt={t[1]-t[0]:.3f})')
print(f'x: [{x_lor.min():.2f}, {x_lor.max():.2f}]  '
      f'y: [{y_lor.min():.2f}, {y_lor.max():.2f}]  '
      f'z: [{z_lor.min():.2f}, {z_lor.max():.2f}]')

## Visualisation

In [ ]:
fig = plt.figure(figsize=(16, 10))
fig.suptitle('Lorenz Attractor', fontsize=14, fontweight='bold')

# 3D attractor
ax3d = fig.add_subplot(2, 3, (1, 4), projection='3d')
ax3d.plot(x_lor, y_lor, z_lor, lw=0.4, alpha=0.8, color='steelblue')
ax3d.set_xlabel('x'); ax3d.set_ylabel('y'); ax3d.set_zlabel('z')
ax3d.set_title('3D Butterfly Attractor')

# Time series
n_plot = 500
for i, (data, label, color) in enumerate([
    (x_lor, 'x(t)', 'steelblue'),
    (y_lor, 'y(t)', 'darkorange'),
    (z_lor, 'z(t)', 'green')
]):
    ax = fig.add_subplot(2, 3, i + 3 + 1)
    ax.plot(t[:n_plot], data[:n_plot], lw=0.8, color=color)
    ax.set_title(label)
    ax.set_xlabel('t')
    ax.set_ylabel(label)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('lorenz_plots.png', dpi=150, bbox_inches='tight')
plt.show()

## Train / Test Split and Prediction Setup

Two common QRC tasks on Lorenz:

**Task 1 -- One-step-ahead prediction** (standard RC benchmark):  
  Reservoir sees x(t), predicts x(t+dt)

**Task 2 -- Autonomous (closed-loop) forecasting** (Gauthier 2021 style):  
  Train on short segment, then run the reservoir autonomously feeding its own predictions back as inputs. Measure how long the attractor geometry is preserved (valid time).

**Task 3 -- Cross-variable prediction**:  
  Reservoir sees x(t) only, predicts y(t) and z(t) -- tests nonlinear state reconstruction.

In [ ]:
# Split: 70% train, 10% val, 20% test
N = len(xyz)
n_train = int(0.70 * N)
n_val   = int(0.10 * N)
n_test  = N - n_train - n_val

xyz_train = xyz[:n_train]
xyz_val   = xyz[n_train:n_train + n_val]
xyz_test  = xyz[n_train + n_val:]

print(f'Train: {xyz_train.shape}  Val: {xyz_val.shape}  Test: {xyz_test.shape}')
print(f'\nTask 1 (1-step prediction):')
print(f'  Input:  xyz[t]       shape {xyz[:-1].shape}')
print(f'  Target: xyz[t+1]     shape {xyz[1:].shape}')

# Lyapunov time: ~1/lambda_max ≈ 1/0.9 ≈ 1.1 time units
lyap_time = 1.0 / 0.9
lyap_steps = int(lyap_time / 0.02)
print(f'\nLyapunov time: {lyap_time:.2f} time units = {lyap_steps} steps')
print('Valid prediction horizon for good models: ~5-10 Lyapunov times')

## Valid Time Metric

For autonomous (closed-loop) Lorenz prediction, the key metric is **valid time**: how long the predicted trajectory stays close to the true trajectory before diverging exponentially.

Standard definition: prediction is valid while error < threshold * std(x)

In [ ]:
def valid_time(y_true, y_pred, threshold=0.4, dt=0.02):
    """
    Compute valid time: time (in Lyapunov units) before prediction error
    exceeds threshold * std(y_true).
    """
    std = np.std(y_true)
    error = np.abs(y_pred - y_true) / std
    exceeds = np.where(error > threshold)[0]
    if len(exceeds) == 0:
        return len(y_true) * dt / (1.0 / 0.9)  # full range
    return exceeds[0] * dt / (1.0 / 0.9)  # in Lyapunov times

print('valid_time() is ready -- use it after running your reservoir predictions.')
print('Published results (Gauthier 2021, Next-Gen RC): valid time ~5-8 Lyapunov times')
print('Classical ESN (1000 nodes):                    valid time ~3-5 Lyapunov times')

## Save Dataset to CSV

In [ ]:
df = pd.DataFrame({'t': t, 'x': x_lor, 'y': y_lor, 'z': z_lor})
df.to_csv('lorenz.csv', index=False)

# Save splits
for name, data, t_seg in [
    ('train', xyz_train, t[:n_train]),
    ('val',   xyz_val,   t[n_train:n_train+n_val]),
    ('test',  xyz_test,  t[n_train+n_val:]),
]:
    pd.DataFrame({'t': t_seg, 'x': data[:, 0], 'y': data[:, 1], 'z': data[:, 2]}).to_csv(
        f'lorenz_{name}.csv', index=False
    )

print('Saved: lorenz.csv, lorenz_train/val/test.csv')
print(f'Total points: {len(df)}')

## Using This Dataset in QRC

```python
# Task: predict [x(t+1), y(t+1), z(t+1)] from [x(t), y(t), z(t)]
# (or predict x(t+1) from x(t) only -- univariate variant)

for t_idx, state in enumerate(xyz_train[:-1]):
    # Encode: use each component as a separate qubit rotation
    # e.g. Rx(pi * x_normalised) tensor Rx(pi * y_normalised) ...
    rho = encode_multivariate(state, n_qubits=N)
    rho = reservoir_evolve(rho)
    features[t_idx] = measure_all(rho)     # shape: (N,) to (N^2,)

# Multi-output ridge regression
W_out = ridge_regression(features, xyz_train[1:], alpha=1e-6)

# Autonomous rollout
state = xyz_test[0]
predictions = []
for _ in range(len(xyz_test)):
    rho = encode_multivariate(state, n_qubits=N)
    rho = reservoir_evolve(rho)
    feat = measure_all(rho)
    state = feat @ W_out     # closed loop
    predictions.append(state)

vt = valid_time(xyz_test[:, 0], np.array(predictions)[:, 0])
print(f'Valid time: {vt:.2f} Lyapunov times')
```

Key references in your collection:
- Fujii & Nakajima (2020): Lorenz attractor in Fig. 3
- Gauthier et al. (2021): Next-generation RC, autonomous Lorenz forecasting